In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch_geometric.data import Data

import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src")))

from graph_utils import *
from utils.utils import *
from train_GNN_coarsening_fans import train_GNN_coarsening_aware_loss

import warnings
warnings.filterwarnings("ignore")

Using device: cuda


In [2]:
def load_amlgentex(name: str = '10k_only_fans') -> dict:
    '''
    Load AMLGentex dataset. Returns a dictionary with:
        - num_nodes: Total number of nodes in the graph
        - num_edges: Total number of edges in the graph
        - edges: Tensor (2, num_edges) with source and destination node indices
        - features: Tensor (num_nodes, 2) with node features
        - weights: Tensor (num_edges,) with edge weights (transaction amounts)
        - edge_features: Tensor (num_edges, 2) with edge features (amount, type)
        - ground_truth: Tensor (num_nodes,) with SAR labels (1 for SAR, 0 otherwise)
        - candidates_sar: List of lists with candidate SAR accounts per alert model
    
    CASH IN and CASH OUT transactions are not included (those having node ID -1 or -2).
    '''

    PATH = './data/AMLGentex/'
    
    # transaction data creation (edges, features, weights, edge_features, ground_truth)
    tx = pd.read_parquet(f'{PATH}{name}/temporal/tx_log.parquet')
    tx = tx[tx['nameOrig'] != -2]
    tx = tx[tx['nameDest'] != -1]
    
    map_type = { 'INITALBALANCE': 0, 'CASH': 1, 'TRANSFER': 2 }
    
    num_edges = len(tx)
    num_nodes = np.concatenate([tx['nameOrig'].unique(), tx['nameDest'].unique()]).max() + 1

    edges = torch.zeros((2, num_edges), dtype=torch.long)
    weights = torch.zeros(num_edges, dtype=torch.float)
    edge_features = torch.zeros((num_edges, 2), dtype=torch.float)
    ground_truth = torch.zeros(num_nodes, dtype=torch.long)
    features = torch.zeros((num_nodes, 2), dtype=torch.float)

    for i, row in enumerate(tx.itertuples()):
        src = row.nameOrig
        dst = row.nameDest
        
        edges[0][i] = src
        edges[1][i] = dst
        
        weights[i] = row.amount

        edge_features[i][0] = row.amount
        edge_features[i][1] = map_type[row.type]

        # if row.isSAR == 1:
        #     ground_truth[dst] = 1
        #     ground_truth[src] = 1

        features[src][0] = int(row.phoneChangesOrig)
        features[src][1] = int(row.daysInBankOrig)
        features[dst][0] = int(row.phoneChangesDest)
        features[dst][1] = int(row.daysInBankDest)

    # alert models candidate creation
    alert_models = pd.read_csv(f'{PATH}{name}/spatial/alert_models.csv')    

    num_sar_models = alert_models['modelID'].unique().shape[0]
    candidates_sar = [[] for _ in range(num_sar_models)]

    for row in alert_models.itertuples():
        candidates_sar[row.modelID].append(row.accountID)
        ground_truth[row.accountID] = 1

    return {
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "edges": edges,
        "features": features,
        "weights": weights,
        "edge_features": edge_features,
        "ground_truth": ground_truth,
        "candidates_sar": candidates_sar
    }

In [3]:
def create_pyg_data(features, edges_idx, edge_features, labels, weights=None) -> Data:
    # edge_index, edge_attr = to_undirected(edges_idx, edge_features)
    if weights is None:
        weights = torch.ones(edges_idx.shape[1], device=labels.device)


    return Data(
        x=features,
        edge_index=edges_idx,
        edge_attr=edge_features,
        y=labels,
        num_nodes=features.shape[0],
        edge_weight=weights
    )


In [4]:
AML_DATASET = load_amlgentex()

EDGE_IDX       = AML_DATASET['edges']          # 2, M 
EDGE_FEATURES  = AML_DATASET['edge_features']  # M, 2
FEATURES       = AML_DATASET['features']       # N, 2
LABELS         = AML_DATASET['ground_truth']   # N,
WEIGHTS        = AML_DATASET['weights']        # M,
CANDIDATE_SAR  = AML_DATASET['candidates_sar'] # list of lists

G = create_pyg_data(
    features=FEATURES,
    edges_idx=EDGE_IDX,
    edge_features=EDGE_FEATURES,
    labels=LABELS,
    weights=WEIGHTS
)

In [5]:

from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import torch_geometric

def load_amlgentex_data(experiment_root: Path, config_dir: Path):
    """
    Load and preprocess AMLGentex dataset.

    Args:
        experiment_root: Path to experiment directory
        config_dir: Path to config directory

    Returns:
        G: PyTorch Geometric Data object
        node_to_index: Dict mapping account IDs to node indices
    """
    from utils.preprocessor import DataPreprocessor
    from utils.config import load_preprocessing_config

    preproc_config = load_preprocessing_config(str(config_dir / "preprocessing.yaml"))

    print("Preprocessing configuration:")
    print(f"  Raw data: {preproc_config['raw_data_file']}")
    print(f"  Output dir: {preproc_config['preprocessed_data_dir']}")
    print("\nPreprocessing transactions...")
    print("Generating temporal features with rolling windows...\n")

    preprocessor = DataPreprocessor(preproc_config)
    datasets = preprocessor(preproc_config["raw_data_file"])

    nodes_df = datasets["trainset_nodes"]
    edges_df = datasets["trainset_edges"]

    # Create node ID to index mapping
    node_to_index = {
        account_id: idx for idx, account_id in enumerate(nodes_df["account"])
    }

    # Get train/val/test indices
    train_idx = torch.tensor(
        [node_to_index[acc] for acc in nodes_df[nodes_df["train_mask"]]["account"]],
        dtype=torch.long,
    )
    val_idx = torch.tensor(
        [node_to_index[acc] for acc in nodes_df[nodes_df["val_mask"]]["account"]],
        dtype=torch.long,
    )
    test_idx = torch.tensor(
        [node_to_index[acc] for acc in nodes_df[nodes_df["test_mask"]]["account"]],
        dtype=torch.long,
    )

    # Prepare features and labels
    nodes_df = nodes_df.drop(columns=["bank"])
    X = (
        nodes_df.drop(
            columns=["account", "train_mask", "val_mask", "test_mask", "is_sar"]
        )
        .to_numpy()
        .astype(np.float32)
    )
    y = nodes_df["is_sar"].to_numpy().astype(np.int64)

    # Map edges
    edges_df["src_idx"] = edges_df["src"].map(node_to_index)
    edges_df["dst_idx"] = edges_df["dst"].map(node_to_index)
    edges_df = edges_df.dropna(subset=["src_idx", "dst_idx"])
    edges = edges_df[["src_idx", "dst_idx"]].to_numpy().astype(np.int64)
    edges_index = torch.tensor(edges.T, dtype=torch.long)

    # Normalize features
    scaler = MinMaxScaler().fit(X)
    X_normalized = torch.tensor(scaler.transform(X), dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.int64)

    # Create PyG Data object
    G = torch_geometric.data.Data(x=X_normalized, edge_index=edges_index, y=y)
    G = torch_geometric.transforms.ToUndirected()(G)
    G.edge_weight = torch.ones(G.edge_index.size(1))
    G.train_idx = train_idx
    G.val_idx = val_idx
    G.test_idx = test_idx
    G.W, G.L, G.dw = graph_params(G)

    return G, node_to_index

In [6]:
experiement_root = Path("./data/AMLGentex/10k_only_fans/")
config_dir = experiement_root / "config"

G, _ = load_amlgentex_data(experiement_root, config_dir)

# method = "variation_edges"
method = "variation_fans"
epochs_per_lev = [10]
# epochs_per_lev = [1, 2, 5, 10]
max_cost_loss = 0.20


for ep_per_lev in epochs_per_lev:

    Gall, Call, iCs = train_GNN_coarsening_aware_loss(
        G,
        levels=25,
        K=20,
        lr=0.01,
        epoch_per_level=ep_per_lev,
        method=method,
        max_cost_loss=max_cost_loss,
        candidates=CANDIDATE_SAR,
    )
    
    name = f"data_gnn_CoarseningAwareLoss_V2_epochs_{ep_per_lev}.npy"
    data = np.load(f"{save_path}{name}", allow_pickle=True).item()
    LOGGER.info(f"Epochs per Level: {ep_per_lev}")
    LOGGER.info(
        f"nodes: {data['Gall'][-1].num_nodes}, edges: {data['Gall'][-1].num_edges}, coarse accuracy: {data['ycrs'][-1]:.4f}, fine accuracy: {data['yfine'][-1]:.4f}"
    )

    # plot iteration vs accuracy
    plt.figure(figsize=(16, 9))
    plt.plot(data["ycrs"], marker="*", label=f"Coarse")
    plt.plot(data["yfine"], linestyle=":", marker="o", label=f"Fine")
    plt.xlabel("Iteration")
    plt.ylabel("Accuracy")
    plt.title("Coarse vs Fine Accuracy over Iterations")
    plt.tight_layout()
    plt.legend()
    plt.savefig(f"{save_path}/iterative_custom_loss_accuracy_{ep_per_lev}.png")

    # plot number of nodes vs accuracy
    plt.figure(figsize=(16, 9))
    plt.plot(np.array(data["num_nodes_coarse"]), data["ycrs"], marker="*", label=f"Coarse")
    plt.plot(np.array(data["num_nodes_coarse"]), data["yfine"], linestyle=":", marker="o", label=f"Fine")
    plt.xlabel("# Nodes Coarse")
    plt.ylabel("Accuracy")
    plt.title("Coarse vs Fine Accuracy over #Nodes")
    plt.tight_layout()
    plt.legend()
    plt.savefig(f"{save_path}/iterative_custom_loss_accuracy_{ep_per_lev}_with_nodes.png")


plt.show()

Preprocessing configuration:
  Raw data: data/AMLGentex/10k_only_fans/temporal/tx_log.parquet
  Output dir: data/AMLGentex/10k_only_fans/preprocessed

Preprocessing transactions...
Generating temporal features with rolling windows...


Preprocessing data...Warning: Windows cover 100/101 steps. Using non-overlapping windows.

Transductive label splitting:
  Total nodes: 9931, SAR nodes: 5177, Normal nodes: 4754
  Train: 3106 SAR + 2852 normal = 5958
  Val:   1035 SAR + 950 normal = 1985
  Test:  1036 SAR + 952 normal = 1988
 done



Coarsening Levels:   0%|          | 0/25 [00:00<?, ?it/s]

: 